# Dr.Copper — Phase 1: Macro PCA Pipeline

In [ ]:
import dr_copper as dc
from pathlib import Path

import warnings

warnings.filterwarnings("ignore")

FIGURES = Path("../outputs/figures")
DATA = Path("../data")

## 1 — Fetch & inspect data

To analyse the copper market, we need to look at other main commodities like gold and oil. As copper is a raw material for the economic engine, it is also reasonable to look at the economic indices of the largest world economies in the world market, like the USA and China.

In [2]:
raw = dc.fetch_raw(start="2010-01-01")

print("Raw shape:", raw.shape)
raw.tail()

Raw shape: (4125, 9)


,crude,gold,copper,dxy,sp500,DFII10,cli_china,cli_usa,USDCNY
date,,,,,,,,,
2026-05-20,98.260002,4531.299805,6.2905,99.110001,7432.970215,2.13,NaN,NaN,6.8145
2026-05-21,96.349998,4539.799805,6.2570,99.190002,7445.720215,2.18,NaN,NaN,6.8005
2026-05-22,96.599998,4521.000000,6.3420,99.320000,7473.470215,2.16,NaN,NaN,6.8025
2026-05-26,93.889999,4500.399902,6.3610,99.150002,7519.120117,NaN,NaN,NaN,6.7945
2026-05-27,89.839996,4473.100098,6.3720,99.056999,7522.580078,NaN,NaN,NaN,6.7777


In [3]:
features = dc.fetch_features(raw)

print("Features shape:", features.shape)
features.tail()

Features shape: (4031, 15)


,crude_ret1d,crude_ret5d,gold_ret1d,gold_ret5d,copper_ret1d,copper_ret5d,dxy_ret1d,dxy_ret5d,sp500_ret1d,sp500_ret5d,copper_vol21d,cli_china,cli_usa,DFII10,USDCNY
date,,,,,,,,,,,,,,,
2026-05-18,0.030271,0.102542,-0.000725,-0.035857,0.003274,-0.022310,-0.003027,0.010462,-0.000736,-0.001322,0.307503,98.801,100.8471,2.13,6.8092
2026-05-19,-0.008224,0.053263,-0.010200,-0.037309,-0.017207,-0.050604,0.003329,0.010223,-0.006701,-0.006418,0.311299,98.801,100.8471,2.18,6.8000
2026-05-20,-0.092382,-0.027701,0.005532,-0.036064,0.020152,-0.053393,-0.001915,0.006377,0.010734,-0.001516,0.317099,98.801,100.8471,2.13,6.8145
2026-05-21,-0.019630,-0.048815,0.001874,-0.030009,-0.005340,-0.048432,0.000807,0.003130,0.001714,-0.007429,0.311652,98.801,100.8471,2.18,6.8005
2026-05-22,0.002591,-0.087374,-0.004150,-0.007668,0.013493,0.014373,0.001310,0.000504,0.003720,0.008731,0.312951,98.801,100.8471,2.16,6.8025


## 2 — Feature matrix preview

In [4]:
features.describe().round(4)

,crude_ret1d,crude_ret5d,gold_ret1d,gold_ret5d,copper_ret1d,copper_ret5d,dxy_ret1d,dxy_ret5d,sp500_ret1d,sp500_ret5d,copper_vol21d,cli_china,cli_usa,DFII10,USDCNY
count,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000,4031.0000
mean,0.0002,0.0007,0.0003,0.0017,0.0001,0.0007,0.0001,0.0003,0.0005,0.0023,0.2252,100.0686,99.8172,0.5657,6.6634
std,0.0254,0.0553,0.0107,0.0232,0.0152,0.0334,0.0045,0.0099,0.0109,0.0224,0.0902,1.2919,0.9755,0.8978,0.3586
min,-0.2822,-0.4818,-0.1207,-0.1444,-0.2517,-0.2882,-0.0240,-0.0442,-0.1277,-0.1980,0.0927,89.6241,92.8922,-1.1900,5.8630
25%,-0.0115,-0.0260,-0.0046,-0.0112,-0.0077,-0.0172,-0.0025,-0.0059,-0.0038,-0.0074,0.1672,99.4932,99.2597,0.0500,6.3561
50%,0.0010,0.0027,0.0004,0.0028,0.0002,0.0009,0.0001,0.0003,0.0007,0.0042,0.2094,100.1345,99.8316,0.5000,6.6594
75%,0.0124,0.0286,0.0059,0.0157,0.0086,0.0202,0.0026,0.0064,0.0057,0.0145,0.2537,100.7758,100.4478,1.2100,6.9435
max,0.2239,0.6883,0.0591,0.1097,0.1244,0.1887,0.0203,0.0529,0.0909,0.1604,1.0173,102.6806,101.3689,2.5200,7.3500


## 3 — Train Test split

In [5]:
split_index = int(len(features) * 0.8)

features_train = features.iloc[:split_index]
features_test = features.iloc[split_index:]

split_date = features_train.index[-1]

raw_train = raw.loc[raw.index <= split_date]
raw_test = raw.loc[raw.index > split_date]

## 4 — Save data (Parquet + CSV)

In [6]:
# --- Parquet (primary, efficient) ---
raw.to_parquet(DATA / "raw" / "raw.parquet")
raw_train.to_parquet(DATA / "raw" / "raw_train.parquet")
raw_test.to_parquet(DATA / "raw" / "raw_test.parquet")

features.to_parquet(DATA / "processed" / "features.parquet")
features_train.to_parquet(DATA / "processed" / "features_train.parquet")
features_test.to_parquet(DATA / "processed" / "features_test.parquet")


# --- CSV snapshot (human-readable backup) ---
features.to_csv(DATA / "processed" / "features.csv")
features_train.to_csv(DATA / "processed" / "features_train.csv")
features_test.to_csv(DATA / "processed" / "features_test.csv")

## 5 — Fit PCA pipeline

In [7]:
pipe, loadings, scores, evr = dc.fit_pca(features_train, n_components=0.6)

print("Pipeline steps :", pipe.steps)
print("Loadings shape :", loadings.shape)
print("Scores shape   :", scores.shape)
print("EVR            :", evr.round(3))
loadings.round(3)

Pipeline steps : [('scaler', StandardScaler()), ('pca', PCA(n_components=0.6, random_state=42))]
Loadings shape : (15, 6)
Scores shape   : (3224, 6)
EVR            : [0.204 0.137 0.104 0.077 0.071 0.066]


,PC1,PC2,PC3,PC4,PC5,PC6
copper_ret1d,0.348,0.086,-0.042,0.386,0.084,0.026
copper_ret5d,0.387,0.078,-0.039,-0.334,0.080,0.016
copper_vol21d,0.031,-0.138,-0.057,-0.032,0.035,0.951
gold_ret1d,0.248,-0.016,0.434,0.353,-0.235,0.121
gold_ret5d,0.274,-0.059,0.444,-0.199,-0.170,0.094
crude_ret1d,0.281,0.054,-0.329,0.381,-0.230,-0.008
crude_ret5d,0.309,0.077,-0.302,-0.190,-0.277,0.016
sp500_ret1d,0.273,0.091,-0.321,0.252,0.275,-0.026
sp500_ret5d,0.330,0.101,-0.294,-0.337,0.189,0.005
dxy_ret1d,-0.301,-0.044,-0.309,-0.269,-0.114,0.104


## 6 — Loadings heatmap

To not overload the notebook with figures we save them into ..outputs/figures/

In [8]:
dc.plot_loadings_heatmap(
    loadings,
    save_path=FIGURES / "pca_loadings_heatmap.png",
)

## 7 — Explained variance (scree)

In [9]:
dc.plot_explained_variance(
    evr,
    save_path=FIGURES / "pca_explained_variance.png",
)

## 8 — PC score time series

In [10]:
dc.plot_pc_scores(
    scores,
    copper_price=raw["copper"],
    save_path=FIGURES / "pca_scores_timeseries.png",
)

After PCA we have 6 pcs which together explain slightly more 65% of variance. First 3 of them explain ~45% of variance, so we are going to use them for regime recognition, and we need to interpret them:

- PC1 has positive loadings on commodity returns and the S&P 500, and a negative loading on the DXY index. This component appears to capture a global liquidity and risk-on factor, where commodity and equity prices tend to rise alongside a weaker U.S. dollar.

- PC2 has positive loadings on both the U.S. and Chinese Composite Leading Indicators (CLI) and a negative loading on USD/CNY. This component can be interpreted as a global growth factor, reflecting synchronized improvement in U.S. and Chinese economic expectations together with yuan strength.

- PC3 has a positive loading on gold returns and negative loadings on crude oil, the S&P 500, and the DXY index. This component appears to represent a defensive factor, characterized by gold outperforming while cyclical and risk-sensitive assets weaken.

## 9 — K-Means regime identification

In [11]:
dc.plot_elbow(
    scores,
    save_path=FIGURES / "kmeans_elbow.png",
)

Using the elbow method, we can simply choose 4 regimes to implement the k-means.

In [12]:
labels, km, summary = dc.fit_regimes(scores, n_regimes=4)
print(summary)

          PC1    PC2    PC3  n_obs
regime                            
0      -2.519 -0.849  0.497    479
1      -0.575  0.895 -0.317   1190
2       0.675 -1.582 -0.404    709
3       1.669  0.548  0.503    846


In short remembering that PC1 - liquidity/risk-on, PC2 - global growth, PC3 - defensive factor we interpet regimes:
 - Regime 0: Really weak liquidity, weak global growth, defensive gold = Risk-off regime or macro stress
 - Regime 1: Weak liquidity, strong global growth, little defense in gold = Recovery
 - Regime 2: Good liquidity, very weak global growth, little defense in gold = Pumping economy with money 
 - Regime 3: Strong liquidity, good global growth, defensive gold as well = Global growth but still with good performing gold as, mb, hedge

In [13]:
dc.plot_regime_history(
    labels,
    copper_price=raw_train["copper"],
    save_path=FIGURES / "kmeans_regime_history.png",
)

In [14]:
dc.plot_regime_scatter(
    scores,
    labels,
    save_path=FIGURES / "kmeans_regime_scatter.png",
)

## 10 — Save pipeline artefacts

In [15]:
import joblib

joblib.dump(pipe, DATA / "processed" / "pca_pipeline.joblib")
joblib.dump(km, DATA / "processed" / "km.joblib")
scores.to_parquet(DATA / "processed" / "pc_scores.parquet")
loadings.to_parquet(DATA / "processed" / "pc_loadings.parquet")
labels.to_frame().to_parquet(DATA / "processed" / "regime_labels.parquet")